# Chapter 6 — Everything is a rule

Step 5 builds the **one pass mechanism**. From here on, every compiler
activity — simplification, backend decompositions, param legalization,
transform columns, even rendering — is `(pattern, fn)` **data**, run by a
single greedy driver. The review question this chapter arms: *any proposed
new mechanism must first prove it cannot be a rule.*

| File | Counted lines / cap | What |
|---|---|---|
| `src/pdum/dsl/kernel/rewrite.py` | 113 / 150 | `Pat`, rule sets, the driver, `Stage` legality, `MatchLog`, the non-termination budget |

The driver's character, on purpose: **greedy, directional, deterministic** —
bottom-up, first-matching-rule-wins (order is priority), DAG sharing
preserved, loud budget guard. The end of this chapter measures the road not
taken (equality saturation as the core) with a real e-graph engine.

Glossary terms settled: *rule / RuleSet / Stage (legality), match log.*

In [1]:
from pdum.dsl.kernel import types as T
from pdum.dsl.kernel.ir import Builder, Region
from pdum.dsl.kernel.ops import CORE_OPS
from pdum.dsl.kernel.printer import print_program
from pdum.dsl.kernel.rewrite import MatchLog, Pat, Stage, rewrite, run_stage

b = Builder(CORE_OPS)


def const(v):
    return b.emit("core.const", type=T.f64, value=v)


def is_const(m, name, value=None):
    n = m[name]
    return n.op == "core.const" and (value is None or dict(n.attrs)["value"] == value)


# Rules are DATA: a pattern and a replacement function. This is the entire
# extension mechanism for compiler behavior.
SIMPLIFY = [
    (Pat("core.add", ("x", "z"), guard=lambda m: is_const(m, "z", 0.0)), lambda bl, m: m["x"]),
    (Pat("core.mul", ("x", "z"), guard=lambda m: is_const(m, "z", 1.0)), lambda bl, m: m["x"]),
    (
        Pat("core.add", ("x", "y"), guard=lambda m: is_const(m, "x") and is_const(m, "y")),
        lambda bl, m: bl.emit(
            "core.const", type=m["root"].type, value=dict(m["x"].attrs)["value"] + dict(m["y"].attrs)["value"]
        ),
    ),
    (Pat("core.neg", (Pat("core.neg", ("x",)),)), lambda bl, m: m["x"]),
    (Pat("core.sub", ("x", "x")), lambda bl, m: bl.emit("core.const", type=m["root"].type, value=0.0)),
]

x = b.emit("core.env", type=T.f64, slot=0)
noisy = b.emit("core.add", b.emit("core.mul", b.emit("core.add", x, const(0.0)), const(1.0)), const(0.0))
before = Region(body=(b.emit("core.yield", noisy),))
print(print_program(before, name="before"))

log = MatchLog()
after = rewrite(before, SIMPLIFY, CORE_OPS, name="simplify", log=log)
print()
print(print_program(after, name="after"))
print()
for stage, old, new in log.entries:
    print(f"  [{stage}] {old.op}  ->  {new.op}")

before() {
  %0 = core.env {slot = 0} : f64
  %1 = core.const {value = 0.0} : f64
  %2 = core.add %0, %1 : f64
  %3 = core.const {value = 1.0} : f64
  %4 = core.mul %2, %3 : f64
  %5 = core.const {value = 0.0} : f64
  %6 = core.add %4, %5 : f64
  core.yield %6
}

after() {
  %0 = core.env {slot = 0} : f64
  core.yield %0
}

  [simplify] core.add  ->  core.env
  [simplify] core.mul  ->  core.env
  [simplify] core.add  ->  core.env


`((x+0)*1)+0` collapsed to `x` in one bottom-up pass — each child's rewrite
exposed the parent's opportunity, and the match log shows every firing. Two
disciplines worth noticing in the driver: **DAG sharing is preserved** (a
shared node is rewritten once, and stays shared), and **binders are
untouchable** (`core.param` is never rewritten — the one structural
exemption).

In [2]:
# Sharing: the SAME node object used twice is rewritten once, and the result
# is still shared. (This is what keeps content keys and artifact reuse sane.)
shared = b.emit("core.add", b.emit("core.env", type=T.f64, slot=1), const(0.0))
root = b.emit("core.mul", shared, shared)
log2 = MatchLog()
out = rewrite(Region(body=(b.emit("core.yield", root),)), SIMPLIFY, CORE_OPS, log=log2)
res = out.body[-1].args[0]
print("still shared:", res.args[0] is res.args[1], "| rule firings:", len(log2.entries))

# Nonlinear patterns: a repeated capture name demands structural equality.
print("x - x -> ", rewrite(
    Region(body=(b.emit("core.yield", b.emit("core.sub", b.emit("core.env", type=T.f64, slot=2),
                                             b.emit("core.env", type=T.f64, slot=2))),)),
    SIMPLIFY, CORE_OPS).body[-1].args[0].attrs)

still shared: True | rule firings: 1
x - x ->  (('value', 0.0),)


In [3]:
# Provenance rides along for free (050_provenance_tracking.md): fresh
# replacement nodes inherit the replaced node's loc via the builder default;
# survivors keep their own story; and type errors render the source points.
from pdum.dsl.kernel.ir import CallLoc, Loc, format_loc

lit = b.emit("core.add", const(3.0), const(4.0), loc=Loc("art.py", 7))
folded = rewrite(Region(body=(b.emit("core.yield", lit),)), SIMPLIFY, CORE_OPS).body[-1].args[0]
print("folded const:", dict(folded.attrs), "carries", format_loc(folded.loc))
print("an inlining chain renders as:", format_loc(CallLoc(Loc("wave.py", 5), Loc("art.py", 40))))

i3 = b.emit("core.env", type=T.i64, slot=3, loc=Loc("art.py", 11))
f4 = b.emit("core.env", type=T.f64, slot=4, loc=Loc("art.py", 12))
try:
    b.emit("core.add", i3, f4, loc=Loc("art.py", 13))
except TypeError as terr:
    print("a typed error is a starting region:")
    print("  ", terr)

folded const: {'value': 7.0} carries art.py:7
an inlining chain renders as: wave.py:5 (inlined from art.py:40)
a typed error is a starting region:
   core arithmetic is strict: i64 vs f64 — insert an explicit core.cast [art.py:13; art.py:11; art.py:12]


In [4]:
# Rules reach inside regions — branches of a core.if are cleaned in place:
e = b.emit("core.env", type=T.f64, slot=0)
cond = b.emit("core.cmp", e, e, pred="lt")
branch = b.emit("core.if", cond, regions=(
    Region(body=(b.emit("core.yield", b.emit("core.add", e, const(0.0))),)),
    Region(body=(b.emit("core.yield", b.emit("core.neg", b.emit("core.neg", e))),)),
))
cleaned = rewrite(Region(body=(b.emit("core.yield", branch),)), SIMPLIFY, CORE_OPS)
print(print_program(cleaned, name="cleaned"))

cleaned() {
  %0 = core.env {slot = 0} : f64
  %1 = core.cmp %0, %0 {pred = 'lt'} : bool
  %2 = core.if %1 ({
    core.yield %0
  }, {
    core.yield %0
  }) : f64
  core.yield %2
}


In [5]:
# Stages: rules to fixpoint, THEN conversion-target legality — "which
# dialects may exist after stage N" is machine-checked, not folklore.
from pdum.dsl.kernel.ir import VerifyError
from pdum.dsl.kernel.ops import OpDef

toy_ops = dict(CORE_OPS)
toy_ops["toy.blit"] = OpDef("toy.blit", lambda a, at, r: T.f64)
tb = Builder(toy_ops)
prog_with_toy = Region(body=(tb.emit("core.yield", tb.emit("toy.blit")),))

try:
    run_stage(prog_with_toy, Stage("mid", [], legal=frozenset({"core"})), toy_ops)
except VerifyError as verr:  # NB: `as e` would DELETE our env node named e — Python quirk
    print("refused:", verr)

# And the non-termination budget — a depth-growing rule fails LOUDLY, not
# with a blown stack:
ping = [(Pat("core.neg", ("x",)), lambda bl, m: bl.emit("core.neg", bl.emit("core.neg", m["root"])))]
try:
    rewrite(Region(body=(b.emit("core.yield", b.emit("core.neg", e)),)), ping, CORE_OPS, name="ping")
except VerifyError as err:
    print("guarded:", err)

refused: [mid] illegal op 'toy.blit'; legal namespaces: ['core']
guarded: [ping] rewrite did not stabilize (budget exhausted at 'core.neg')


## The detour, measured: should equality saturation be the core?

The honest question (asked at this step's walkthrough): e-graphs solve the
**phase-ordering problem** — instead of greedily committing to each rewrite,
an e-graph holds *all* equivalent forms at once and extracts the best by
cost. Is that technology strong enough to BE the engine, rather than a
satellite? We installed `egglog` (the state-of-the-art e-graph engine, Rust
core) and measured, rather than argued:

In [6]:
import time

try:
    t0 = time.perf_counter()
    from egglog import (  # noqa: I001
        EGraph, Expr, StringLike, birewrite, i64, i64Like, rewrite as eg_rw, ruleset, var, vars_,
    )

    IMPORT_MS = 1000 * (time.perf_counter() - t0)

    class Num(Expr):
        def __init__(self, value: i64Like) -> None: ...
        @classmethod
        def var(cls, name: StringLike) -> Num: ...
        def __add__(self, other: Num) -> Num: ...
        def __mul__(self, other: Num) -> Num: ...

    a2, b2, c2 = vars_("a b c", Num)
    i2, j2 = var("i", i64), var("j", i64)
    algebra = ruleset(
        eg_rw(a2 + Num(0)).to(a2),
        eg_rw(a2 * Num(1)).to(a2),
        eg_rw(Num(i2) + Num(j2)).to(Num(i2 + j2)),
        eg_rw(Num(i2) * Num(j2)).to(Num(i2 * j2)),
        birewrite(a2 * (b2 + c2)).to(a2 * b2 + a2 * c2),  # distribution, BOTH ways
        birewrite(a2 + b2).to(b2 + a2),
        birewrite(a2 * b2).to(b2 * a2),
    )

    # The phase-ordering win our greedy driver CANNOT get: x*2 + x*3 -> x*5
    # requires FACTORING (reverse distribution) — a greedy engine with
    # distribute-forward walks away from it. The e-graph holds both.
    eg = EGraph()
    xx = Num.var("x")
    expr = eg.let("e", xx * Num(2) + xx * Num(3))
    t0 = time.perf_counter()
    eg.run(algebra * 8)
    t1 = time.perf_counter()
    print("extracted:", eg.extract(expr))
    print(f"import={IMPORT_MS:.0f}ms  saturate={1000 * (t1 - t0):.1f}ms")
except ImportError:
    print("egglog not installed on this platform — see the measured summary below")

extracted: Num.var("x") * Num(5)
import=95ms  saturate=0.9ms


**The verdict** (recorded in 010 §10, confirming research verdicts V2/R7 —
now with our own numbers instead of citations):

| Axis | Greedy driver (ours) | Equality saturation (egglog) |
|---|---|---|
| Phase-ordering | loses (commits greedily) | **wins** (`x*2+x*3 → x*5`, ~1 ms) |
| Kernel-sized cost | microseconds | **~20 ms** saturate+extract — our whole miss budget |
| Determinism | total order, golden-testable | bounded iterations ⇒ heuristic output |
| Non-equational passes (slot numbering, AD, render) | natural | not equalities — poor fit |
| Binders (`core.for`/`core.call` params) | untouched by design | the classic e-graph hard case |
| Dependency | zero (stdlib) | Rust wheel, **~1.5 s import** |

**Decision: the greedy driver is the core; equality saturation is an
opt-in optimizer pass** at the seam that already exists (a pass is any
`Region → Region` function — architecture §12). Where factoring-class
optimizations matter, an egglog-backed pass exports a pure expression
subregion, saturates, extracts by cost, and rebuilds — paying its ~20 ms
only where a domain asked for it. What we deliberately give up in the core:
the phase-ordering wins. What we keep: determinism, zero dependencies, and
one engine a reader can hold in their head — all 113 lines of it.

## What we can't do yet

- Nobody *feeds* the engine: `lower_ast` rules and the compile driver's
  stage ladder arrive with lowering (ch07); today stages are assembled by
  hand.
- Decompositions gated on backend op sets and param legalization (the
  `abi.slot` stage) arrive with marshaling and the first backends
  (ch08–ch10).
- Transform columns (jvp/batch) are the same rule shape, much later.

**Gates armed:** golden-printed-IR-after-stage (`tests/test_rewrite.py`).

**Budget after step 5:** kernel 707 / 1150 counted lines.

**Next:** ch07 — source to IR: `classify_names`, the fused typing+lowering
pass, the combinator build rule, and the dialect cast-insertion decision.